In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W


C:\spark-3.5.4-bin-hadoop3


In [2]:
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "midterm1",     # your MySQL database
    "conn_props" : {
        "user" : "root",
        "password" : "1234",    # your MySQL password
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}
mongodb_args = {
    "cluster_location" : "atlas",
    "user_name" : "linj716",
    "password" : "Northwind2024",
    "cluster_name" : "sandbox",
    "cluster_subnet" : "hkabmqp",
    "db_name" : "final_nosql",
    "collection" : "dim_metadata",
    "null_column_threshold" : 0.5
}
# Base project directory
base_dir = os.path.join(os.getcwd(), 'precinct_project')

# Data directories
data_dir = os.path.join(base_dir, 'data')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

os.makedirs(batch_dir, exist_ok=True)
os.makedirs(stream_dir, exist_ok=True)

# Raw streaming JSON input directory
precinct_stream_dir = os.path.join(stream_dir, "precincts")
os.makedirs(precinct_stream_dir, exist_ok=True)

# Spark SQL warehouse directories
dest_database = "precinct_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

# Bronze / Silver / Gold output directories
precinct_output_bronze = os.path.join(database_dir, 'fact_precinct_edits', 'bronze')
precinct_output_silver = os.path.join(database_dir, 'fact_precinct_edits', 'silver')
precinct_output_gold   = os.path.join(database_dir, 'fact_precinct_edits', 'gold')

os.makedirs(precinct_output_bronze, exist_ok=True)
os.makedirs(precinct_output_silver, exist_ok=True)
os.makedirs(precinct_output_gold, exist_ok=True)


In [3]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
    print(f"The stream has processed {len(query.recentProgress)} batches")


def remove_directory_tree(path: str):
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
    except Exception as e:
        return f"An error occurred: {e}"


def drop_null_columns(df, threshold):
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    return df_dropped
    

def get_mysql_dataframe(spark_session, sql_query : str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    
    dframe = spark_session.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("driver", args['conn_props']['driver']) \
        .option("user", args['conn_props']['user']) \
        .option("password", args['conn_props']['password']) \
        .option("query", sql_query) \
        .load()
    
    return dframe
    

def get_mongo_uri(**args):
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = (
            f"mongodb+srv://{args['user_name']}:{args['password']}@"
            f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
            f"{args['db_name']}"
        )
    else:
        uri = f"mongodb://localhost:27017/{args['db_name']}"

    return uri

def get_spark_conf_args(spark_jars : list, **args):
    jars = ", ".join(spark_jars)
    
    sparkConf_args = {
        "app_name" : "PySpark Precinct Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars,
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
        .setMaster(args['worker_threads']) \
        .set('spark.driver.memory', '4g') \
        .set('spark.executor.memory', '2g') \
        .set('spark.jars', args['spark_jars']) \
        .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1') \
        .set('spark.mongodb.input.uri', args['mongo_uri']) \
        .set('spark.mongodb.output.uri', args['mongo_uri']) \
        .set('spark.sql.adaptive.enabled', 'false') \
        .set('spark.sql.debug.maxToStringFields', 35) \
        .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
        .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
        .set('spark.sql.streaming.schemaInference', 'true') \
        .set('spark.sql.warehouse.dir', args['database_dir']) \
        .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    
    return sparkConf


def get_mongo_client(**args):
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())
    else:
        client = pymongo.MongoClient(mongo_uri)
    return client
    

def get_mongodb_dataframe(spark_session, **args):
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()

    dframe = dframe.drop('_id')
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    
    return dframe

def set_mongo_collections(mongo_client, db_name: str, data_directory: str, json_files: dict):
    """
    Load JSON files into MongoDB collections.
    json_files should be a dict: {collection_name: filename}
    """
    db = mongo_client[db_name]
    
    for collection_name, filename in json_files.items():
        # Drop existing collection
        db.drop_collection(collection_name)
        
        # Load JSON file
        json_path = os.path.join(data_directory, filename)
        with open(json_path, 'r') as openfile:
            json_object = json.load(openfile)
        
        # Insert into MongoDB
        collection = db[collection_name]
        collection.insert_many(json_object)
    
    mongo_client.close()


In [4]:
remove_directory_tree(database_dir)


"Directory 'C:\\Users\\linj7\\Downloads\\Midterm1 SQL\\spark-warehouse\\precinct_dlh.db' has been removed successfully."

In [5]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
jars.append(mysql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)
sparkConf = get_spark_conf(**sparkConf_args)

spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark


In [6]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Final Project Database'
    WITH DBPROPERTIES (contains_pii = false, purpose = 'DS-2002 Final Project');
"""
spark.sql(sql_create_db)


DataFrame[]

In [7]:
spark.read.format("jdbc") \
    .option("url", f"jdbc:mysql://{mysql_args['host_name']}:{mysql_args['port']}/{mysql_args['db_name']}") \
    .option("driver", mysql_args['conn_props']['driver']) \
    .option("user", mysql_args['conn_props']['user']) \
    .option("password", mysql_args['conn_props']['password']) \
    .option("dbtable", "information_schema.tables") \
    .load() \
    .filter("table_schema = 'midterm1'") \
    .select("table_name") \
    .show(100, truncate=False)


+-------------+
|table_name   |
+-------------+
|dim_areas    |
|dim_date     |
|dim_metadata |
|dim_precincts|
+-------------+



In [8]:
sql_dim_precincts = "SELECT * FROM dim_precincts"
sql_dim_area      = "SELECT * FROM dim_areas"
sql_dim_date      = "SELECT * FROM dim_date"

df_dim_precincts = get_mysql_dataframe(spark, sql_dim_precincts, **mysql_args)
df_dim_area      = get_mysql_dataframe(spark, sql_dim_area, **mysql_args)
df_dim_date      = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

df_dim_precincts.show(5)
df_dim_area.show(5)
df_dim_date.show(5)


+------------+--------+--------------+--------------+
|precinct_key|OBJECTID|  PrecinctName|PrecinctNumber|
+------------+--------+--------------+--------------+
|           1|       1|   Jackson-Via|           301|
|           2|       2|        Buford|           303|
|           3|       3|  Summit/Clark|           102|
|           4|       4|Key Recreation|           101|
|           5|       5|           CHS|           401|
+------------+--------+--------------+--------------+
only showing top 5 rows

+--------+--------+------------------+----------------+
|area_key|OBJECTID|       ShapeSTArea|   ShapeSTLength|
+--------+--------+------------------+----------------+
|       1|       1|3.20470692208264E7|36433.3122138073|
|       2|       2|1.94430765460801E7|  20543.63525975|
|       3|       3|2.56025827345768E7| 25249.436474963|
|       4|       4|3.73986514756506E7|30639.3458440959|
|       5|       5|6.53180650740686E7|56452.4900896593|
+--------+--------+------------------+---

In [9]:
client = get_mongo_client(**mongodb_args)

data_dir = os.path.join(os.getcwd(), "data")
json_files = {"dim_metadata": "metadata.json"}   # collection_name : file_name

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)


In [10]:
df_dim_metadata = get_mongodb_dataframe(
    spark_session=spark,
    **mongodb_args
)

df_dim_metadata.show(5)


+--------+--------------------+----------------+
|OBJECTID|    last_edited_date|last_edited_user|
+--------+--------------------+----------------+
|       1|2023/09/26 17:48:...|        WINKLERM|
|       2|2023/09/26 17:37:...|        WINKLERM|
|       3|2024/06/18 18:13:...|            CITY|
|       4|2024/01/27 08:12:...|        WINKLERM|
|       5|2023/09/26 17:37:...|        WINKLERM|
+--------+--------------------+----------------+
only showing top 5 rows



In [11]:
df_dim_precincts.write.saveAsTable(f"{dest_database}.dim_precincts", mode="overwrite")
df_dim_area.write.saveAsTable(f"{dest_database}.dim_area", mode="overwrite")
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")
df_dim_metadata.write.saveAsTable(f"{dest_database}.dim_metadata", mode="overwrite")


In [13]:
get_file_info(precinct_stream_dir)


,name,size,modification_time
0,precinct_edits_01.json,255,2026-04-30 00:29:51.777824879
1,precinct_edits_02.json,255,2026-04-30 00:30:36.576484919
2,precinct_edits_03.json,307,2026-04-30 00:30:50.910574675


In [15]:
df_precinct_bronze = (
    spark.readStream
        .option("schemaLocation", precinct_output_bronze)
        .option("maxFilesPerTrigger", 1)
        .option("multiLine", "true")
        .json(precinct_stream_dir)
)

df_precinct_bronze.isStreaming

True

In [16]:
precinct_checkpoint_bronze = os.path.join(precinct_output_bronze, "_checkpoint")

bronze_query = (
    df_precinct_bronze
        .withColumn("receipt_time", current_timestamp())
        .withColumn("source_file", input_file_name())
        .writeStream
        .format("parquet")
        .outputMode("append")
        .queryName("precinct_bronze")
        .trigger(availableNow=True)
        .option("checkpointLocation", precinct_checkpoint_bronze)
        .option("compression", "snappy")
        .start(precinct_output_bronze)
)

In [17]:
print(f"Query ID: {bronze_query.id}")
print(f"Query Name: {bronze_query.name}")
print(f"Query Status: {bronze_query.status}")

Query ID: 907336de-4203-4701-af67-feab82422c98
Query Name: precinct_bronze
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [18]:
bronze_query.awaitTermination()

In [20]:
df_dim_edit_date = df_dim_date.select(
    col("date_key").alias("edit_date_key"),
    col("date").alias("edit_full_date")
)

df_dim_metadata = df_dim_metadata.withColumnRenamed("metadata_id", "metadata_no")

In [26]:
bronze = spark.readStream.format("parquet").load(precinct_output_bronze).alias("b")
precincts = df_dim_precincts.alias("p")
areas = df_dim_area.alias("a")
metadata = df_dim_metadata.alias("m")
dates = df_dim_edit_date.alias("d")

df_precinct_silver = (
    bronze

        # Join to precinct dimension
        .join(precincts, col("b.OBJECTID") == col("p.OBJECTID"))

        # Join to area dimension
        .join(areas, col("b.OBJECTID") == col("a.OBJECTID"))

        # Join to metadata dimension
        .join(metadata, col("b.OBJECTID") == col("m.OBJECTID"), "left_outer")

        # Join to date dimension
        .join(
            dates,
            dates.edit_full_date.cast(DateType()) ==
            col("b.EditTimestamp").cast(DateType()),
            "left_outer"
        )

        # Final Silver schema
        .select(
            col("b.OBJECTID").cast(LongType()).alias("object_id"),
            col("b.PrecinctName").alias("precinct_name"),
            col("b.PrecinctNumber").cast(IntegerType()).alias("precinct_number"),
            col("b.EditType").alias("edit_type"),
            col("b.EditTimestamp").alias("edit_timestamp"),
            col("b.User").alias("edit_user"),
            col("b.Notes").alias("edit_notes"),

            # Surrogate keys
            col("p.precinct_key").cast(LongType()).alias("precinct_key"),
            col("a.area_key").cast(LongType()).alias("area_key"),
            col("d.edit_date_key").cast(LongType()).alias("edit_date_key"),

            # Metadata attributes
            col("m.last_edited_user"),
            col("m.last_edited_date")
        )
)

df_precinct_silver.isStreaming

True

In [27]:
precinct_checkpoint_silver = os.path.join(precinct_output_silver, "_checkpoint")

precinct_silver_query = (
    df_precinct_silver.writeStream
        .format("parquet")
        .outputMode("append")
        .queryName("precinct_silver")
        .trigger(availableNow=True)
        .option("checkpointLocation", precinct_checkpoint_silver)
        .option("compression", "snappy")
        .start(precinct_output_silver)
)



In [28]:
print(f"Query ID: {precinct_silver_query.id}")
print(f"Query Name: {precinct_silver_query.name}")
print(f"Query Status: {precinct_silver_query.status}")


Query ID: e06c3caf-846a-4cbf-bf3b-02435bd06263
Query Name: precinct_silver
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [29]:
precinct_silver_query.awaitTermination()

In [30]:
df_precinct_gold = (
    spark.readStream
        .format("parquet")
        .load(precinct_output_silver)
        .select(
            col("object_id").cast(LongType()),
            col("precinct_key").cast(LongType()),
            col("area_key").cast(LongType()),
            col("edit_date_key").cast(LongType()),

            col("precinct_name").cast(StringType()),
            col("precinct_number").cast(IntegerType()),
            col("edit_type").cast(StringType()),
            col("edit_timestamp").cast(TimestampType()),
            col("edit_user").cast(StringType()),
            col("edit_notes").cast(StringType()),

            col("last_edited_user").cast(StringType()),
            col("last_edited_date").cast(StringType())
        )
)

df_precinct_gold.isStreaming


True

In [31]:
precinct_checkpoint_gold = os.path.join(precinct_output_gold, "_checkpoint")

precinct_gold_query = (
    df_precinct_gold.writeStream
        .format("parquet")
        .outputMode("append")
        .queryName("precinct_gold")
        .trigger(availableNow=True)
        .option("checkpointLocation", precinct_checkpoint_gold)
        .option("compression", "snappy")
        .start(precinct_output_gold)
)


In [ ]:
df_gold.orderBy("year", "month", desc("edit_events")).show()


In [32]:
print(f"Query ID: {precinct_gold_query.id}")
print(f"Query Name: {precinct_gold_query.name}")
print(f"Query Status: {precinct_gold_query.status}")

Query ID: 744fb603-b0af-4f60-81d2-1404904b5a64
Query Name: precinct_gold
Query Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [33]:
precinct_gold_query.awaitTermination()

In [34]:
get_file_info(precinct_output_gold)


,name,size,modification_time
0,.part-00000-1440968d-6e9f-4008-bcb9-e48989af6a...,40,2026-04-30 00:59:18.132047415
1,part-00000-1440968d-6e9f-4008-bcb9-e48989af6a2...,3898,2026-04-30 00:59:18.133045911


In [35]:
print("dim_precincts:", df_dim_precincts.count())
print("dim_area:", df_dim_area.count())
print("dim_date:", df_dim_date.count())


dim_precincts: 9
dim_area: 9
dim_date: 3


In [36]:
df_dim_precincts.printSchema()
df_dim_area.printSchema()
df_dim_date.printSchema()


root
 |-- precinct_key: integer (nullable = true)
 |-- OBJECTID: integer (nullable = true)
 |-- PrecinctName: string (nullable = true)
 |-- PrecinctNumber: integer (nullable = true)

root
 |-- area_key: integer (nullable = true)
 |-- OBJECTID: integer (nullable = true)
 |-- ShapeSTArea: double (nullable = true)
 |-- ShapeSTLength: double (nullable = true)

root
 |-- date_key: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- quarter: integer (nullable = true)



In [37]:
print("dim_metadata:", df_dim_metadata.count())


dim_metadata: 9


In [38]:
df_dim_metadata.printSchema()
df_dim_metadata.show(5)


root
 |-- OBJECTID: integer (nullable = true)
 |-- last_edited_date: string (nullable = true)
 |-- last_edited_user: string (nullable = true)

+--------+--------------------+----------------+
|OBJECTID|    last_edited_date|last_edited_user|
+--------+--------------------+----------------+
|       1|2023/09/26 17:48:...|        WINKLERM|
|       2|2023/09/26 17:37:...|        WINKLERM|
|       3|2024/06/18 18:13:...|            CITY|
|       4|2024/01/27 08:12:...|        WINKLERM|
|       5|2023/09/26 17:37:...|        WINKLERM|
+--------+--------------------+----------------+
only showing top 5 rows



In [39]:
get_file_info(precinct_output_bronze)


,name,size,modification_time
0,.part-00000-2503aa75-1397-4c54-b1d7-195dd81b19...,44,2026-04-30 00:34:12.161948681
1,.part-00000-ce7e4bad-ab72-4381-a73a-965ab9053b...,48,2026-04-30 00:34:13.991111040
2,.part-00000-e472f924-6e64-423c-b7f3-6a561ac3df...,44,2026-04-30 00:34:10.156490326
3,part-00000-2503aa75-1397-4c54-b1d7-195dd81b19b...,4559,2026-04-30 00:34:12.162951231
4,part-00000-ce7e4bad-ab72-4381-a73a-965ab9053b2...,4804,2026-04-30 00:34:13.991111040
5,part-00000-e472f924-6e64-423c-b7f3-6a561ac3df3...,4526,2026-04-30 00:34:10.156490326
